In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from itertools import chain

from response_stats.generators import PoissonSpikeGenerator
from response_stats.responses import ResponseConfig, make_response_data, ResponseDetector
from response_stats.responses.response_tests import WilcoxonTest, SurrogateTest
from response_stats.pipeline import *

from response_stats.plot_utils import *

import matplotlib.pyplot as plt

from scipy.stats import pearsonr

data_dir = Path("/media/al/darch/response_stats/datasets/test/test.parquet")

In [ ]:
def _parse_cli_args(argv= None) -> Dict[str, Any]:
    import argparse

    p = argparse.ArgumentParser(description="Run spike simulation pipeline.")
    p.add_argument("--config", type=Path, required=True,
                   help="Path to YAML configuration file.")
    args = p.parse_args(list(argv) if argv is not None else None)
    return {"config_path": args.config}

def _parse_save_by_language(cfg, save_path, df, fname):
    if cfg.save_language == "python":
        df.to_parquet(save_path / f"{fname}.parquet")
    elif cfg.save_language == "matlab":
        mat_dict = {col: df[col].to_numpy() for col in df.columns}
        savemat(save_path / f"{fname}.mat", {"data": mat_dict})

def _copy_config_file(config_path, save_path):
    shutil.copy(config_path, f"{save_path}/run_config.yaml")

In [ ]:
config_path = Path("/home/al/Documents/code/generate_responses/configs/main_dataset/[51,75]_response.yaml")
cfg = SimulationConfig.from_yaml(config_path)

rs = ResponseSimulator(cfg)

In [ ]:
response_bool = 1 if cfg.response_type == "response" else 0

In [24]:
parent_path = Path("/home/al/Documents/code/generate_responses/configs/main_dataset")

line_number = 1          # 1-based line number


old_string =  "10"
new_string =  "5"

for p in parent_path.rglob("*"):

    # Read file
    with p.open("r", encoding="utf-8") as f:
        lines = f.readlines()

    # Modify specific line
    idx = line_number - 1
    if old_string in lines[idx]:
        lines[idx] = lines[idx].replace(old_string, new_string)

    # Write back
    with p.open("w", encoding="utf-8") as f:
        f.writelines(lines)

In [ ]:
parent_path = Path("/home/al/Documents/code/generate_responses/configs/main_dataset")

line_number = 1            # 1-based line number
old_value = 15000
new_value = 1000

old_string = str(old_value)
new_string = str(new_value)

for p in parent_path.rglob("*"):

    # Read file
    with p.open("r", encoding="utf-8") as f:
        lines = f.readlines()

    # Modify specific line
    idx = line_number - 1
    if old_string in lines[idx]:
        lines[idx] = lines[idx].replace(old_string, new_string)

    # Write back
    with p.open("w", encoding="utf-8") as f:
        f.writelines(lines)

In [ ]:
fr_baseline = rs._handle_baseline_firing_rates()
trial_counts = rs._handle_trial_counts()

fr_response, beta_a_s, beta_b_s = rs._handle_response_type(trial_counts, fr_baseline)